## Zero Shot GeoCIR using G3

In [1]:
import sys

sys.path.append("..")
import os
import torch
from tqdm import tqdm
import polars as pl
from src.g3 import G3
from PIL import Image
import json
import torch.nn.functional as F
import matplotlib.pyplot as plt
from src.utils import read_index
import numpy as np
from torch import nn
from src.metrics import evaluate
from src.utils import sum_compose

device = "cuda" if torch.cuda.is_available() else "cpu"
base_img_path = "../../datasets/google-landmark/index-img"
topk = 100
batch_size = 128

In [2]:
model = G3().to(device)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
index, meta = read_index("../index/g3_index_gld_siglip_filtered")
index.hnsw.efSearch = 128
meta = meta["metadata"]

In [4]:
test_set = json.loads(open("../../datasets/geocir-triplet/test.v1.json", "r").read())
test_set = test_set["data"]

In [5]:
df_test = pl.read_csv("../../datasets/google-landmark/index_ref_predicted_siglip_filtered.csv")

In [23]:
all_gt = [b["target_img_ids"] for b in test_set]
query_img_ids = [b["ref_img_id"] for b in test_set]
all_pred = []
latlons = df_test[["latitude", "longitude"]].to_numpy()

img_embs = []
txt_embs = []
loc_embs = []
img2text_embs = []
img2loc_embs = []


for i in tqdm(range(0, len(test_set), batch_size), desc="batch test"):
    batch = test_set[i : i + batch_size]
    img_inputs = [os.path.join(base_img_path, f"{b['img_id']}.jpg") for b in batch]
    captions = [b["caption"] for b in batch]
    latlon = torch.tensor(latlons[[i["ref_img_id"] for i in batch]])

    with torch.no_grad():
        img_processed = model.preprocess_image(img_inputs)
        text_processed = model.preprocess_text(captions)

        img_emb = model.vision_proj(model.vision_model(img_processed.to(device)).pooler_output)
        img_emb_norm = F.normalize(img_emb, dim=-1)

        text_emb = model.text_proj(model.text_model(**{k: v.to(device) for k, v in text_processed.items()}).pooler_output)
        text_emb_norm = F.normalize(text_emb, dim=-1)

        query_emb_norm = sum_compose(img_emb, text_emb, alpha=1, beta=2)

        img2text_emb = model.img2txt_proj(img_emb)
        img2text_emb_norm = F.normalize(img2text_emb, dim=-1)

        loc_emb = model.loc2img_proj(model.location_encoder(latlon))
        loc_emb_norm = F.normalize(loc_emb, dim=-1)

        img2loc_emb = model.img2loc_proj(img_emb)
        img2loc_emb_norm = F.normalize(img2loc_emb, dim=-1)

        query = torch.cat([query_emb_norm, img2text_emb_norm, loc_emb_norm], dim=1).cpu().numpy()

        img_embs.append(img_emb)
        txt_embs.append(text_emb)
        loc_embs.append(loc_emb)
        img2text_embs.append(img2text_emb)
        img2loc_embs.append(img2loc_emb)

    # retrieve
    sim, ind = index.search(query, topk + 1)
    all_pred.append(ind)

all_pred = np.vstack(all_pred)

evaluate(all_pred, all_gt, query_img_ids)

batch test: 100%|██████████| 128/128 [03:53<00:00,  1.83s/it]


{'mAP@5': 0.022155659378564646,
 'mAP@10': 0.022098294211596193,
 'mAP@25': 0.023922324301106003,
 'mAP@50': 0.025350448860964932,
 'mAP@100': 0.026709979537096806}

In [24]:
img_embs = F.normalize(torch.vstack(img_embs), dim=-1)
txt_embs = F.normalize(torch.vstack(txt_embs), dim=-1)
loc_embs = F.normalize(torch.vstack(loc_embs), dim=-1)
img2text_embs = F.normalize(torch.vstack(img2text_embs), dim=-1)
img2loc_embs = F.normalize(torch.vstack(img2loc_embs), dim=-1)

In [36]:
import torch.nn.functional as F
# composed = img_embs - 5 * loc_embs + txt_embs
# composed = F.normalize(composed, dim=-1).numpy()
# _, ind = index.search(composed, 101)
# evaluate(ind, all_gt, query_img_ids)

visual = img_embs + 2 * txt_embs
text = img2text_embs # + txt_embs
loc = img2loc_embs # - loc_embs
composed = torch.concat([visual, text, loc], dim=1).cpu().numpy()
_, ind = index.search(composed, 101)
evaluate(ind, all_gt, query_img_ids)

{'mAP@5': 0.04287055251600036,
 'mAP@10': 0.042809456490373085,
 'mAP@25': 0.04557814664761376,
 'mAP@50': 0.047873422820212605,
 'mAP@100': 0.049954135884788906}